<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-solution/blob/main/notebooks/EcoMarket_AI_Solution.ipynb)


# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Taller Práctico #1
## Optimización de Atención al Cliente con IA Generativa

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

**Modelo:** `llama-3.3-70b-versatile` via [Groq API](https://console.groq.com)


In [8]:
import warnings
import os
from pathlib import Path
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages

# Base del repositorio: en local es ../ desde el notebook, en Colab es el CWD
if IN_COLAB:
    BASE_DIR = Path(".")
else:
    BASE_DIR = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path("..").resolve()

DATA_DIR    = BASE_DIR / "data"
PROMPTS_DIR = BASE_DIR / "prompts"

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Base dir: {BASE_DIR.resolve()}")

Entorno: Local
Base dir: /Users/gome33773/Documents/MIAA/IAGenerativa/ecomarket-solution


In [9]:
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/requirements.txt -O requirements.txt && \
    pip install -q -r requirements.txt
else:
    print("Entorno local — instala con: pip install -r requirements.txt")

Entorno local — instala con: pip install -r requirements.txt


Cargar data

In [10]:
if IN_COLAB:
    !mkdir -p data
    ![ ! -f data/orders_database.txt ] && wget -q -O data/orders_database.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/data/orders_database.txt
    ![ ! -f data/return_policy.txt ]   && wget -q -O data/return_policy.txt   https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/data/return_policy.txt
    print("Datos descargados desde GitHub")
else:
    print(f"Datos locales en: {DATA_DIR}")

Datos locales en: /Users/gome33773/Documents/MIAA/IAGenerativa/ecomarket-solution/data


Cargar prompts

In [11]:
if IN_COLAB:
    !mkdir -p prompts
    ![ ! -f prompts/system_prompts.txt ]      && wget -q -O prompts/system_prompts.txt      https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/system_prompts.txt
    ![ ! -f prompts/order_status_prompt.txt ]  && wget -q -O prompts/order_status_prompt.txt  https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/order_status_prompt.txt
    ![ ! -f prompts/return_policy_prompt.txt ] && wget -q -O prompts/return_policy_prompt.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/return_policy_prompt.txt
    print("Prompts descargados desde GitHub")
else:
    print(f"Prompts locales en: {PROMPTS_DIR}")

Prompts locales en: /Users/gome33773/Documents/MIAA/IAGenerativa/ecomarket-solution/prompts


In [12]:
# ============================================================
# CONFIGURACIÓN SEGURA GROQ (LOCAL + COLAB)
# ============================================================

import os

def get_api_key():
    """
    Obtiene la API Key de forma segura.
    1) Intenta desde Colab Secrets (si estamos en Colab).
    2) Si no, intenta desde variable de entorno / archivo .env local.
    """
    # --- Intento 1: Colab Secrets ---
    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get("GROQ_API_KEY")
            if api_key:
                print("🔑 API Key cargada desde Colab Secrets")
                return api_key
        except Exception:
            pass

    # --- Intento 2: Variable de entorno / .env local ---
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    api_key = os.getenv("GROQ_API_KEY")
    if api_key:
        print("🔑 API Key cargada desde variable de entorno / .env")
        return api_key

    raise ValueError(
        "No se encontró GROQ_API_KEY.\n"
        "  • En Colab: añádela en Secrets (🔑 icono en el panel izquierdo).\n"
        "  • En local: crea un archivo .env con GROQ_API_KEY=gsk_..."
    )

# ========================
# VARIABLES DE CONFIG
# ========================

GROQ_API_KEY = get_api_key()

CONFIG = {
    "model": "llama-3.3-70b-versatile",
    "api_url": "https://api.groq.com/openai/v1/chat/completions",
    "max_tokens": 1024,
    "temperature": 0.3,
    "timeout": 30
}

# ========================
# HEADERS LISTOS
# ========================

HEADERS = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

print(f"API Key válida ({GROQ_API_KEY[:6]}****)")
print(f"Modelo configurado: {CONFIG['model']}")

🔑 API Key cargada desde variable de entorno / .env
API Key válida (gsk_9V****)
Modelo configurado: llama-3.3-70b-versatile


In [13]:
# Carga de datos de ordenes y politicas

with open(DATA_DIR / "orders_database.txt", "r", encoding="utf-8") as f:
    ORDERS_DATABASE = f.read()

with open(DATA_DIR / "return_policy.txt", "r", encoding="utf-8") as f:
    RETURN_POLICY = f.read()

print("Datos cargados")

Datos cargados


In [14]:
# System Prompts

with open(PROMPTS_DIR / "system_prompts.txt", "r", encoding="utf-8") as f:
    ECOBOT_BASE = f.read()

with open(PROMPTS_DIR / "order_status_prompt.txt", "r", encoding="utf-8") as f:
    ORDER_SYSTEM = ECOBOT_BASE + "\n\n" + f.read()

with open(PROMPTS_DIR / "return_policy_prompt.txt", "r", encoding="utf-8") as f:
    RETURN_SYSTEM = ECOBOT_BASE + "\n\n" + f.read()

print("System prompts cargados")

System prompts cargados


In [15]:
# ============================================================
#  Cliente LLM
# ============================================================

import requests

def call_llm(system_prompt: str, user_message: str) -> str:
    """Llama a la API de Groq y retorna la respuesta generada."""

    if not GROQ_API_KEY:
        return "No se encontró la API Key. Revisa Colab Secrets."

    payload = {
        "model": CONFIG["model"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        "max_tokens": CONFIG["max_tokens"],
        "temperature": CONFIG["temperature"]
    }

    try:
        r = requests.post(
            CONFIG["api_url"],
            headers=HEADERS,
            json=payload,
            timeout=CONFIG["timeout"]
        )

        # ========================
        # MANEJO DE ERRORES
        # ========================
        if r.status_code == 401:
            return "API Key inválida. Verifica en console.groq.com"

        if r.status_code != 200:
            return f"Error HTTP {r.status_code}: {r.text[:200]}"

        data = r.json()

        # ========================
        # VALIDACIÓN RESPUESTA
        # ========================
        if "choices" not in data or len(data["choices"]) == 0:
            return "Respuesta inválida del modelo"

        return data["choices"][0]["message"]["content"].strip()

    except requests.exceptions.Timeout:
        return "Timeout: el modelo tardó demasiado en responder"

    except requests.exceptions.ConnectionError:
        return "Error de conexión: revisa tu internet"

    except Exception as e:
        return f"Error: {str(e)}"


print("Cliente LLM listo (adaptado a CONFIG)")

Cliente LLM listo (adaptado a CONFIG)


# Motor de Prompt Engineering: Consultas y Devoluciones

In [16]:
#  Funciones de Prompt Engineering

def query_order_status(tracking_number: str) -> str:
    """
    Consulta el estado de un pedido.
    Técnicas: RAG simulado, instrucciones paso a paso,
    manejo de casos extremos, restricción anti-alucinación.
    """
    user_prompt = f"""--- BASE DE DATOS DE PEDIDOS (usa ÚNICAMENTE esta información) ---
{ORDERS_DATABASE}
--- FIN DEL CONTEXTO ---

El cliente pregunta por su pedido: {tracking_number}

INSTRUCCIONES:
PASO 1 — Busca '{tracking_number}' en el contexto (mayúsculas/minúsculas, con y sin guión).

PASO 2.1 — Si el pedido EXISTE:
  - Estado actual con descripción clara
  - Fecha estimada de entrega o confirmación si ya fue entregado
  - URL de tracking si está disponible
  - Si RETRASADO: disculpa sincera primero, luego nueva fecha y razón
  - Si CANCELADO: estado del reembolso y plazos exactos
  - Si PENDIENTE DE PAGO: urgencia amable + enlace de pago
  - Próximo paso que el cliente debe esperar

PASO 2.1 — Si el pedido NO APARECE en el contexto:
  - Informa amablemente que no puedes encontrar ese número
  - Sugiere verificar el número (puede haber un error tipográfico)
  - Ofrece contacto: soporte@ecomarket.com o 900-ECO-MKT
  - NUNCA inventes datos de un pedido que no aparece en el contexto

FORMATO: Saludo personalizado + información + oferta de ayuda adicional."""
    return call_llm(ORDER_SYSTEM, user_prompt)


def request_return(product_name: str, reason: str, days: int,
                   opened: bool = False, first_return: bool = False) -> str:
    """
    Procesa una solicitud de devolución.
    Técnicas: árbol de decisión explícito, empatía antes de negativa,
    excepciones documentadas, nunca un 'no' sin alternativa.
    """
    opened_txt = "Sí" if opened else "No especificado"
    first_txt  = "Sí (posible excepción hasta 45 días)" if first_return else "No"

    user_prompt = f"""--- POLÍTICA DE DEVOLUCIONES DE ECOMARKET ---
{RETURN_POLICY}
--- FIN DE LA POLÍTICA ---

SOLICITUD DEL CLIENTE:
  Producto: {product_name}
  Motivo: {reason}
  Días desde la entrega: {days}
  ¿Producto abierto?: {opened_txt}
  ¿Primera devolución del cliente?: {first_txt}

ÁRBOL DE DECISIÓN (sigue en orden):

PASO 1 — Excepción universal:
  ¿El motivo menciona daño en envío, defecto o producto incorrecto?
  -> SÍ: APRUEBA siempre. Cubre el envío. Ofrece reembolso O reenvío. FIN.
  -> NO: continúa al Paso 2.

PASO 2 — Categoría del producto:
  ¿Es alimento, bebida, planta, suplemento? -> NO posible (sanitario). -> Paso 5.
  ¿Es higiene personal Y fue abierto? -> NO posible. -> Paso 5.
  ¿Es personalizado o digital? -> NO posible. -> Paso 5.
  Ninguna aplica -> puede devolverse. -> Paso 3.

PASO 3 — Plazo de 30 días:
  ¿Más de 30 días Y es primera devolución? -> Escala a agente humano, excepción posible.
  ¿Más de 30 días sin excepción? -> NO posible. -> Paso 5.
  ¿Dentro de 30 días? -> Devolución aprobada. -> Paso 4.

PASO 4 — RESPUESTA AFIRMATIVA:
  Confirma, explica el proceso en 3 pasos numerados, plazo del reembolso,
  ofrece crédito de tienda (+5% de bonificación).

PASO 5 — NEGATIVA EMPÁTICA:
  1. Empatía genuina primero (reconoce la frustración)
  2. Razón de forma simple y humana (no párrafo legal)
  3. Al menos UNA alternativa concreta
  4. NUNCA termines con un 'no' definitivo y frío sin ofrecer algo"""
    return call_llm(RETURN_SYSTEM, user_prompt)


def print_response(title: str, response: str):
    print(f"\n{'='*65}")
    print(f"📋 {title}")
    print('='*65)
    print(f"\n🤖 EcoBot:\n")
    print(response)
    print()

print("Funciones de prompt engineering listas")

Funciones de prompt engineering listas


---
## Ejercicio 1 — Consultas de Estado de Pedido
Demuestra el manejo de 5 estados distintos y la protección anti-alucinación.

In [17]:
# Pedido EN TRÁNSITO — información de entrega y tracking
response = query_order_status("ECO-12345")
print_response("ECO-12345 — Pedido EN TRÁNSITO — información de entrega y tracking", response)


📋 ECO-12345 — Pedido EN TRÁNSITO — información de entrega y tracking

🤖 EcoBot:

Hola María,

Me alegra poder ayudarte con tu consulta sobre el estado de tu pedido ECO-12345. Después de verificar en nuestra base de datos, puedo informarte que tu pedido está actualmente EN TRÁNSITO. Esto significa que tu Kit de bambú reutilizable (3 piezas) ya ha sido despachado y se encuentra en camino hacia ti.

La fecha estimada de entrega para tu pedido es el 2024-07-05. Puedes seguir el progreso de tu pedido a través del enlace de seguimiento proporcionado por DHL Express: https://track.dhl.com/ECO12345. Nuestro último punto de control indica que tu pedido pasó por el Centro logístico Madrid.

No hay retrasos reportados en tu pedido, así que deberías recibirlo según lo programado. Si tienes alguna otra pregunta o necesitas ayuda adicional, no dudes en preguntar. Estoy aquí para ayudarte.

¿Hay algo más en lo que pueda asistirte hoy?



In [18]:
# Pedido RETRASADO — debe incluir disculpa empática primero
response = query_order_status("ECO-12346")
print_response("ECO-12346 — Pedido RETRASADO — debe incluir disculpa empática primero", response)


📋 ECO-12346 — Pedido RETRASADO — debe incluir disculpa empática primero

🤖 EcoBot:

Hola Carlos,

Entiendo que estás preocupado por el estado de tu pedido #ECO-12346. Desafortunadamente, he encontrado que tu pedido se encuentra retrasado. Quiero disculparme sinceramente por las molestias que esto te puede estar causando. La nueva fecha estimada de entrega es el 2024-07-08, y el retraso se debe a una huelga de transporte regional en Cataluña.

Puedes seguir el estado de tu pedido en el enlace de seguimiento: https://track.correos.es/ECO12346. Nuestro equipo está trabajando para minimizar el impacto de este retraso y asegurarse de que tu pedido llegue lo antes posible.

Si tienes alguna otra pregunta o necesitas ayuda adicional, no dudes en preguntar. Estoy aquí para ayudarte. También puedes contactar con nuestro equipo de soporte en soporte@ecomarket.com o llamando al 900-ECO-MKT (gratuito, L-V 9:00-18:00h) si necesitas más información o asistencia.

¿Hay algo más en lo que pueda ayuda

In [19]:
# Pedido ENTREGADO — Confirmación de entrega
response = query_order_status("ECO-12347")
print_response("ECO-12347 — Pedido ENTREGADO — Confirmación de entrega", response)


📋 ECO-12347 — Pedido ENTREGADO — Confirmación de entrega

🤖 EcoBot:

Hola Ana Martínez,

Me alegra poder informarte sobre el estado de tu pedido ECO-12347. Según nuestros registros, tu pedido ha sido **ENTREGADO** con éxito. La entrega se realizó el 2024-06-25 a las 14:32h, y se ha registrado una firma digital como confirmación de recepción.

Puedes verificar el estado de tu pedido en el enlace de seguimiento: https://track.seur.com/ECO12347. Si tienes alguna pregunta o inquietud sobre tu pedido, no dudes en hacérmelo saber.

En EcoMarket, nos comprometemos a brindarte la mejor experiencia posible y a ser transparentes en todo momento. Si necesitas ayuda adicional o tienes alguna otra pregunta, estoy aquí para ayudarte. ¿Hay algo más en lo que pueda asistirte?



In [30]:
# Pedido CANCELADO — debe informar el estado del reembolso
response = query_order_status("ECO-12349")
print_response("ECO-12349 — Pedido CANCELADO — debe informar el estado del reembolso", response)


📋 ECO-12349 — Pedido CANCELADO — debe informar el estado del reembolso

🤖 EcoBot:

Hola Sofia,

Me dirijo a ti porque he encontrado información sobre un pedido a tu nombre. El estado actual de tu pedido ECO-12349 es CANCELADO. La cancelación se realizó el 2024-07-01 a petición tuya. En cuanto al reembolso, te informo que ya ha sido procesado el 2024-07-02. Debes recibir el reembolso en un plazo de 5-7 días hábiles en la tarjeta original terminada en ****4521.

Si tienes alguna pregunta adicional o necesitas más información sobre el proceso de reembolso, no dudes en hacérmelo saber. Estoy aquí para ayudarte.

¿Hay algo más en lo que pueda asistirte? Puedes responder a este mensaje o, si lo prefieres, contactar con nuestro equipo de soporte a través de soporte@ecomarket.com o llamando al 900-ECO-MKT. Estamos a tu disposición.



In [31]:
# Pedido PENDIENTE DE PAGO — urgencia amable
response = query_order_status("ECO-12351")
print_response("ECO-12351 — Pedido PENDIENTE DE PAGO — urgencia amable", response)


📋 ECO-12351 — Pedido PENDIENTE DE PAGO — urgencia amable

🤖 EcoBot:

Hola Isabella,

Me alegra que hayas contactado con nosotros sobre tu pedido ECO-12351. Desafortunadamente, he encontrado que tu pedido está en estado "PENDIENTE DE PAGO". Esto significa que el pago no fue confirmado y es importante que completes el pago lo antes posible para evitar que tu pedido sea cancelado automáticamente.

Para completar el pago, puedes acceder al enlace de pago siguiente: https://ecomarket.com/pago/ECO12351. Es importante que completes el pago en las próximas 24 horas para evitar cualquier retraso en la entrega de tu pedido.

Si tienes alguna pregunta o necesitas ayuda con el proceso de pago, no dudes en hacérmelo saber. Estoy aquí para ayudarte. Además, si necesitas cualquier asistencia adicional, puedes contactar con nuestro equipo de soporte en soporte@ecomarket.com o llamando al 900-ECO-MKT.

¿Hay algo más en lo que pueda ayudarte hoy? Estoy aquí para asegurarme de que tengas una experiencia

In [32]:
# Número INEXISTENTE — prueba anti-alucinación
response = query_order_status("ECO-99999")
print_response("ECO-99999 — Número INEXISTENTE — prueba anti-alucinación", response)


📋 ECO-99999 — Número INEXISTENTE — prueba anti-alucinación

🤖 EcoBot:

Hola, lamentablemente no puedo encontrar el pedido con el número ECO-99999 en nuestra base de datos. Es posible que haya un error tipográfico en el número de pedido que me has proporcionado. Te sugiero verificar el número de pedido nuevamente para asegurarte de que es correcto.

Si sigues teniendo problemas para encontrar tu pedido, no dudes en contactarnos. Puedes escribirnos un correo electrónico a soporte@ecomarket.com o llamarnos al 900-ECO-MKT (gratuito, de lunes a viernes de 9:00 a 18:00 horas). Estamos aquí para ayudarte.

¿Hay algo más en lo que pueda asistirte mientras tanto? Estoy aquí para ayudarte con cualquier pregunta o inquietud que tengas sobre tus pedidos o nuestros productos.



---
## Ejercicio 2 — Solicitudes de Devolución
El desafío: responder con empatía tanto el «sí» como el «no»,
y nunca dar una negativa sin ofrecer una alternativa.

In [33]:
# Devolución APROBADA — producto de hogar dentro del plazo
response = request_return(product_name="Botella de acero inoxidable 750ml", reason="No me gusta el diseño", days=12, opened=False)
print_response("Devolución APROBADA — producto de hogar dentro del plazo", response)


📋 Devolución APROBADA — producto de hogar dentro del plazo

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver la botella de acero inoxidable 750ml porque no te gusta el diseño. Me parece completamente comprensible que quieras un producto que se adapte a tus gustos y necesidades.

Después de revisar nuestra política de devoluciones, puedo confirmar que la botella de acero inoxidable se encuentra dentro de las categorías que pueden ser devueltas. Como has recibido el producto hace solo 12 días, estás dentro del plazo de 30 días naturales para solicitar la devolución.

Aquí te explico el proceso de devolución en 3 pasos sencillos:

1. **Solicita la devolución**: Ya has dado el primer paso al contactar con nosotros. Ahora, te pedimos que accedas a nuestra página web en [ecomarket.com/devoluciones](http://ecomarket.com/devoluciones) y sigas las instrucciones para solicitar la devolución.
2. **Recibe la etiqueta de envío**: En un plazo de 24 horas hábiles, r

In [34]:
# Devolución RECHAZADA — higiene personal abierto
response = request_return(product_name="Jabón orgánico artesanal", reason="No me gusta el olor", days=5, opened=True)
print_response("Devolución RECHAZADA — higiene personal abierto", response)


📋 Devolución RECHAZADA — higiene personal abierto

🤖 EcoBot:

Entiendo perfectamente tu situación, te has comprado un jabón orgánico artesanal y no te gusta el olor. Me imagino que debe ser frustrante recibir un producto que no se ajusta a tus expectativas.

La razón por la que no podemos aceptar la devolución en este caso es que el producto pertenece a la categoría de higiene personal y ha sido abierto. Nuestra política de devoluciones establece que no podemos aceptar devoluciones de productos de higiene personal que han sido abiertos, ya que no podemos garantizar su seguridad e higiene para otros clientes.

Sin embargo, no quiero dejar la situación así. Te ofrezco una alternativa: podrías considerar donar el producto a alguien que pueda apreciarlo o encontrarle un uso alternativo. También te puedo ofrecer un descuento del 10% en tu próxima compra en EcoMarket, para que puedas encontrar un producto que se ajuste mejor a tus necesidades.

Si prefieres, también puedo ofrecerte la opció

In [35]:
# Devolución RECHAZADA — producto perecedero (alimento)
response = request_return(product_name="Mix de frutos secos orgánicos", reason="Cambié de opinión", days=3, opened=False)
print_response("Devolución RECHAZADA — producto perecedero (alimento)", response)


📋 Devolución RECHAZADA — producto perecedero (alimento)

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver un producto y te ayudaré en lo que pueda.

Desafortunadamente, el producto que has solicitado devolver es un mix de frutos secos orgánicos, lo que cae dentro de la categoría de alimentos. Nuestra política de devoluciones establece que no podemos aceptar devoluciones de productos perecederos una vez enviados, por razones de seguridad alimentaria e higiene.

Entiendo perfectamente tu situación y me gustaría ayudarte a encontrar una solución. En este caso, no podemos ofrecer una devolución del producto, pero te puedo ofrecer un descuento o cupón de compensación para tu próxima compra. Esto te permitirá disfrutar de nuestros productos sostenibles y ecológicos en el futuro.

Si deseas, también puedo ofrecerte ayuda para encontrar un producto similar que se adapte a tus necesidades y preferencias. Nuestro equipo de atención al cliente está aquí para ayu

In [38]:
# Devolución APROBADA — daño en tránsito (excepción universal)
response = request_return(product_name="Cepillo de dientes de bambú", reason="El paquete llegó aplastado y roto", days=4, opened=True)
print_response("Devolución APROBADA — daño en tránsito (excepción universal)", response)


📋 Devolución APROBADA — daño en tránsito (excepción universal)

🤖 EcoBot:

¡Hola! Entiendo perfectamente tu situación y me doy cuenta de que el cepillo de dientes de bambú llegó dañado, lo que es muy frustrante. Me imagino cómo te sentirías al recibir un producto en mal estado.

En este caso, como el producto llegó dañado durante el envío, **APRUEBO** la devolución. Quiero asegurarme de que estés satisfecho con tu compra y que recibas un producto en perfectas condiciones.

Aquí te explico el proceso de devolución en 3 pasos numerados:

1. **Solicita la devolución**: Ya has iniciado este proceso, pero si necesitas ayuda, puedes acceder a nuestra página de devoluciones en ecomarket.com/devoluciones.
2. **Recibe la etiqueta de envío**: Te enviaremos una etiqueta de envío prepagada por email en un plazo de 24 horas hábiles. Esto te permitirá enviar el producto de vuelta a nosotros sin costo adicional.
3. **Envía el producto**: Por favor, empaca el producto en su embalaje original si es po

In [39]:
# Devolución FUERA DE PLAZO — primera vez, escala a agente
response = request_return(product_name="Set de cubiertos de bambú", reason="Me lo regalaron y no lo necesito", days=38, opened=False, first_return=True)
print_response("Devolución FUERA DE PLAZO — primera vez, escala a agente", response)


📋 Devolución FUERA DE PLAZO — primera vez, escala a agente

🤖 EcoBot:

Hola, gracias por considerar EcoMarket para tus necesidades de productos sostenibles. Me doy cuenta de que estás en una situación un poco complicada con el set de cubiertos de bambú que te regalaron y que no necesitas. Entiendo perfectamente que a veces recibimos regalos que, aunque bienintencionados, no se ajustan a nuestras necesidades.

En cuanto a la devolución, veo que el producto en cuestión es un set de cubiertos de bambú, lo que significa que cae dentro de la categoría de productos que sí pueden devolverse, siempre y cuando estén sin usar y en su embalaje original. Sin embargo, también veo que han pasado 38 días desde la entrega, lo que supera el plazo estándar de 30 días para devoluciones.

Dado que es tu primera devolución y considerando que el producto no es de las categorías prohibidas, podemos considerar una excepción. Me gustaría escalar tu caso a uno de nuestros agentes humanos para que puedan evalua

---
## Modo Interactivo
Cambia los valores de las variables y ejecuta las celdas para probar con tus propios datos.

In [40]:
# CONSULTA DE PEDIDO PERSONALIZADA
# Pedidos disponibles: ECO-12345 al ECO-12354, o prueba uno inexistente
MI_PEDIDO = "ECO-12353"  # <- cambia aquí

response = query_order_status(MI_PEDIDO)
print_response(f"Consulta personalizada: {MI_PEDIDO}", response)


📋 Consulta personalizada: ECO-12353

🤖 EcoBot:

Hola Valentina,

Me alegra poder ayudarte con tu consulta sobre el pedido ECO-12353. Después de verificar en nuestra base de datos, encontré que tu pedido está en el estado "LISTO PARA RECOGIDA EN TIENDA". Esto significa que tu pedido ya está preparado y listo para que lo recojas en nuestra tienda física ubicada en C/ Gran Vía 45, Local 2, en Madrid.

La recogida está disponible hasta el 2024-07-10, y puedes pasar por la tienda de lunes a sábado entre las 10:00 y las 20:00 horas. Por favor, no olvides llevar contigo tu DNI o el número de pedido para agilizar el proceso.

Si tienes alguna pregunta o necesitas ayuda adicional, no dudes en hacérmelo saber. Estoy aquí para ayudarte. También puedes contactar con nuestro equipo de soporte a través del correo electrónico soporte@ecomarket.com o llamando al 900-ECO-MKT, si lo prefieres.

¿Hay algo más en lo que pueda ayudarte hoy?



In [41]:
# SOLICITUD DE DEVOLUCIÓN PERSONALIZADA
MI_PRODUCTO  = "Velas aromáticas de cera de soja"  # <- cambia aquí
MI_MOTIVO    = "No me gusta el aroma"               # <- cambia aquí
DIAS         = 8                                    # <- días desde la entrega
FUE_ABIERTO  = False                                # <- True o False
PRIMERA_VEZ  = False                                # <- True si es primera devolución

response = request_return(MI_PRODUCTO, MI_MOTIVO, DIAS, FUE_ABIERTO, PRIMERA_VEZ)
print_response(f"Devolución personalizada: {MI_PRODUCTO}", response)


📋 Devolución personalizada: Velas aromáticas de cera de soja

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver las velas aromáticas de cera de soja porque no te gusta el aroma. Me parece completamente comprensible que quieras cambiar de opinión sobre un producto que no se ajusta a tus preferencias.

Después de revisar nuestra política de devoluciones, me alegra informarte que, en este caso, la devolución es posible. Las velas aromáticas de cera de soja entran dentro de la categoría de productos que pueden devolverse, siempre y cuando estén en su embalaje original y no hayan sido utilizadas.

Aquí te explico el proceso de devolución en 3 pasos sencillos:

1. **Solicita la devolución**: Accede a nuestra página web en [ecomarket.com/devoluciones](http://ecomarket.com/devoluciones), introduce tu número de pedido y el motivo de la devolución.
2. **Recibe la etiqueta de envío**: En un plazo de 24 horas hábiles, recibirás una etiqueta de envío prepagada por 